# Semantic Search System

## Objective
The objective of this project is to build a small semantic search system
that uses embeddings and ChromaDB to retrieve relevant document chunks
based on the meaning of a user's query.

The system:
1. Loads three sample documents.
2. Converts the documents into text.
3. Splits the documents into meaningful chunks.
4. Generates embeddings for the chunks.
5. Stores the embeddings in ChromaDB.
6. Accepts a user's natural-language question.
7. Finds the most semantically similar document chunks.
8. Compares semantic search with keyword search.

## ChromaDB
An open-source, local vector database built to store embeddings and quickly find the most similar ones to a query. 

## sentence_transformers 
A Python library that converts text into embedding vectors capturing actual meaning, not just keywords. We're using it because it runs fully offline with no API key or cost.

In [2]:
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
documents_path = Path("documents")

print("Documents folder exists:", documents_path.exists())

Documents folder exists: True


In [4]:
document_files = sorted(documents_path.glob("*.txt"))

print("Documents found:", len(document_files))

for file_path in document_files:
    print( file_path.name)

Documents found: 3
doc1.txt
doc2.txt
doc3.txt


In [5]:
documents = []

for file_path in document_files:
    text = file_path.read_text(encoding="utf-8")
    
    documents.append({
        "filename": file_path.name,
        "text": text
    })

print("Documents loaded successfully.")

Documents loaded successfully.


In [6]:
for document in documents:
    print()
    print("SOURCE:", document["filename"])
    print("CHARACTERS:", len(document["text"]))
    print()
    print(document["text"][:1000])
    print()


SOURCE: doc1.txt
CHARACTERS: 1392

Digital Service Delivery in Punjab: E-Rozgaar and Citizen Facilitation Centers

The Punjab Information Technology Board has established a network of Citizen Facilitation Centers across the province to bring government services closer to the public. These centers allow citizens to apply for domicile certificates, birth registrations, and land record verification without visiting multiple offices. The E-Rozgaar program, a flagship freelance training initiative, has trained thousands of young people in web development, graphic design, and digital marketing, enabling them to earn income through online freelance platforms. Applicants register through an online portal, complete a skills assessment, and are enrolled in structured training batches lasting three to six months. Graduates receive certification and are connected to freelancing marketplaces. The program has been particularly impactful in smaller cities where traditional employment opportunities a

## CHUNKING

Chunking means splitting documents into pieces. We cannot just dump entire document into embedding model and expect one useful vector , a whole document usually covers alot of different topics so its meaning gets blurry. Chunking means breaking documents into smaller pieces , focused pieces first  each piece has one clear, specific meaning that can be embedded accurately.We use paragraph based chunking.

## TYPES
Fixed-size chunking : Divide the text after fixed number of words and characters.

Sentence-based chunking : Divides the text after each sentence.

Paragraph-based chunking : Divide the text whenever there is new paragraph.

Recursive/semantic chunking : It dividesthe text according to meaning and topic.


In [7]:
chunks = []

for document in documents:
    paragraphs = [
        paragraph.strip()
        for paragraph in document["text"].split("\n\n")
        if paragraph.strip()
    ]
    
    for paragraph in paragraphs:
        chunks.append({
            "chunk_id": f"chunk_{len(chunks)}",
            "source": document["filename"],
            "text": paragraph
        })

print("Total chunks created:", len(chunks))

Total chunks created: 6


## INSPECT CHUNKS

In [8]:
for chunk in chunks:
    print()
    print("Chunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Text:", chunk["text"][:500])
    print()


Chunk ID: chunk_0
Source: doc1.txt
Text: Digital Service Delivery in Punjab: E-Rozgaar and Citizen Facilitation Centers


Chunk ID: chunk_1
Source: doc1.txt
Text: The Punjab Information Technology Board has established a network of Citizen Facilitation Centers across the province to bring government services closer to the public. These centers allow citizens to apply for domicile certificates, birth registrations, and land record verification without visiting multiple offices. The E-Rozgaar program, a flagship freelance training initiative, has trained thousands of young people in web development, graphic design, and digital marketing, enabling them to ea


Chunk ID: chunk_2
Source: doc2.txt
Text: Land Record Digitization and the Role of Blockchain in Punjab


Chunk ID: chunk_3
Source: doc2.txt
Text: Punjab's land record system, historically maintained through handwritten patwari registers, has undergone a significant digital transformation through the Land Records Management Informat

## EMBEDDINGS

Embedding means converting text into numbers. Here we use all-MiniLM-L6-v2 Sentence Transformer model. This model converts each text into numerical vector called embedings.These embeddings allow the system to compare the semantic meaning of a user's question with the meaning of document chunks.


In [23]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Embedding model loaded successfully.


## GENERATE EMBEDDINGS


In [25]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Number of chunks:", len(chunk_texts))
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Number of chunks: 6
Number of embeddings: 6
Embedding dimensions: 384


## CHROMA DB

A normal database can store rows and columns and lets you search by exact matches. Chromadb is a vector based database , it is specifically built to store these number-list. Chromadb gives you two choices PersistentClient an Client. Client (in memory ) when you restarts the notebook everything dissapears , Persistent_client saves everything on disk.Here we use PersistentClient.

In [11]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

print("ChromaDB initialized successfully.")

ChromaDB initialized successfully.


In [12]:
collection = chroma_client.get_or_create_collection(
    name="semantic_search_collection"
)

print("ChromaDB collection created.")

ChromaDB collection created.


## METADATA

Meta data provides additional information about each chunk. Metadata allows the search system to tell the user where a retrieved chunk came from.

In [13]:
ids = [
    chunk["chunk_id"]
    for chunk in chunks
]

metadatas = [
    {
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"]
    }
    for chunk in chunks
]

In [14]:
collection.add(
    ids=ids,
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Chunks stored in ChromaDB:", collection.count())

Chunks stored in ChromaDB: 6


## SEMANTIC SEARCH 

Semantic search retrieves information based on meaning instead of exact keyword match. In semantic_search function  takes users question converts it into embeding and then search and return the most relevant chunks while display_semantic_results makes the result readable for user.

In [15]:
def semantic_search(query, top_k=3, max_distance=1.2):
    """
    Search the document collection using semantic similarity.

    Only results with a distance below the maximum
    allowed distance are returned.
    """

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )


    filtered_documents = []
    filtered_metadatas = []
    filtered_distances = []

    for i, distance in enumerate(results["distances"][0]):

        if distance <= max_distance:
            filtered_documents.append(
                results["documents"][0][i]
            )

            filtered_metadatas.append(
                results["metadatas"][0][i]
            )

            filtered_distances.append(distance)

    return {
        "documents": [filtered_documents],
        "metadatas": [filtered_metadatas],
        "distances": [filtered_distances]
    }

In [16]:
def display_semantic_results(query, top_k=3):
    
    results = semantic_search(query, top_k)

    print("QUERY:")
    print(query)

    if len(results["documents"][0]) == 0:
        print("\nNo relevant information found in the documents.")
        return

    for i, document in enumerate(results["documents"][0]):

        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]

        print(f"\nRESULT {i + 1}")
        print("Source:", metadata["source"])
        print("Chunk ID:", metadata["chunk_id"])
        print("Distance:", round(distance, 4))

        print("\nRetrieved text:")
        print(document[:200] + "...")

In [17]:
query = "How can blockchain improve land record management?"

display_semantic_results(query, top_k=3)

QUERY:
How can blockchain improve land record management?

RESULT 1
Source: doc2.txt
Chunk ID: chunk_2
Distance: 0.664

Retrieved text:
Land Record Digitization and the Role of Blockchain in Punjab...

RESULT 2
Source: doc2.txt
Chunk ID: chunk_3
Distance: 0.7924

Retrieved text:
Punjab's land record system, historically maintained through handwritten patwari registers, has undergone a significant digital transformation through the Land Records Management Information System. O...


In [18]:
test_queries = [
    "How can blockchain make land records more secure?",
    "How can AI improve government services?",
    "How can digital technology make government services easier for citizens?",
    "How can technology reduce fraud in government records?",
    "How can AI reduce the workload of government employees?"
]

for query in test_queries:


    print()
    print("QUERY:", query)
    print()

    print("\nSEMANTIC SEARCH")

    display_semantic_results(query, top_k=2)


QUERY: How can blockchain make land records more secure?


SEMANTIC SEARCH
QUERY:
How can blockchain make land records more secure?

RESULT 1
Source: doc2.txt
Chunk ID: chunk_2
Distance: 0.7326

Retrieved text:
Land Record Digitization and the Role of Blockchain in Punjab...

RESULT 2
Source: doc2.txt
Chunk ID: chunk_3
Distance: 0.838

Retrieved text:
Punjab's land record system, historically maintained through handwritten patwari registers, has undergone a significant digital transformation through the Land Records Management Information System. O...

QUERY: How can AI improve government services?


SEMANTIC SEARCH
QUERY:
How can AI improve government services?

RESULT 1
Source: doc3.txt
Chunk ID: chunk_5
Distance: 0.9712

Retrieved text:
Government bodies in Pakistan, including provincial IT boards, have begun exploring generative AI tools to improve internal efficiency and citizen-facing services. Common early use cases include autom...

RESULT 2
Source: doc3.txt
Chunk ID: chunk_4


## INTERACTIVE SEARCH 

Here, the user can enter its own question.The system converts the question into an embedding, searches ChromaDB for semantically similar chunks, and displays the most relevant results.

In [64]:
user_query = input("Ask your question: ")

display_semantic_results(
    user_query,
    top_k=3
)

QUERY:
what is the total gp f pakistan

No relevant information found in the documents.


## KEYWORD SEARCH

Keyword search retrieves documents based on matching words.



In [19]:
def keyword_search(query, top_k=3):
    """
    Perform a simple keyword-based search.
    
    The score is the number of unique query words
    that also appear in the document chunk.
    """
    
    query_words = set(query.lower().split())
    
    results = []
    
    for chunk in chunks:
        chunk_words = set(chunk["text"].lower().split())
        
        matching_words = query_words.intersection(chunk_words)
        score = len(matching_words)
        
        if score > 0:
            results.append({
                "score": score,
                "source": chunk["source"],
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"]
            })
    
    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )
    
    return results[:top_k]

In [20]:
def display_keyword_results(query, top_k=3):
    results = keyword_search(query, top_k)
    
    print("QUERY:")
    print(query)
    print()
    
    if not results:
        print("No keyword matches found.")
        return
    
    for i, result in enumerate(results):
        print(f"\nRESULT {i + 1}")
        print()
        print("Keyword score:", result["score"])
        print("Source:", result["source"])
        print("Chunk ID:", result["chunk_id"])
        print("\nRetrieved text:")
        print(result["text"])

## COMPARISON SEMANNTIC SEARCH VS KEYWORD SEARCH 

We use the same query for both approaches.This allows us to compare how traditional keyword matching and semantic similarity retrieve information differently.

In [21]:
comparison_query = "How can AI improve public sector services?"

print()
print("KEYWORD SEARCH")
print()

display_keyword_results(
    comparison_query,
    top_k=3
)

print("\n\n")
print()
print("SEMANTIC SEARCH")
print()

display_semantic_results(
    comparison_query,
    top_k=3
)


KEYWORD SEARCH

QUERY:
How can AI improve public sector services?


RESULT 1

Keyword score: 2
Source: doc3.txt
Chunk ID: chunk_4

Retrieved text:
Artificial Intelligence Adoption in Pakistan's Public Sector

RESULT 2

Keyword score: 2
Source: doc3.txt
Chunk ID: chunk_5

Retrieved text:
Government bodies in Pakistan, including provincial IT boards, have begun exploring generative AI tools to improve internal efficiency and citizen-facing services. Common early use cases include automated drafting of routine correspondence, summarization of lengthy policy documents for quick executive review, and chatbot assistants that answer frequently asked questions about government procedures. Internship and training programs have introduced young professionals to prompt engineering, retrieval-augmented generation, and embedding-based semantic search as foundational skills for building AI-powered internal tools. A recurring theme in these pilot projects is the importance of data privacy when proce

## several comparisons

In [22]:
comparison_queries = [
    "How can blockchain make land records more secure?",
    "How can AI improve public sector services?",
    "How can digital technology help citizens access government services?"
]
for query in comparison_queries:

    
    print()
    print("QUERY:", query)
    print()

    print("\nKEYWORD SEARCH")
    print()

    display_keyword_results(
        query,
        top_k=2
    )

    print("\nSEMANTIC SEARCH")
    print()

    display_semantic_results(
        query,
        top_k=2
    )


QUERY: How can blockchain make land records more secure?


KEYWORD SEARCH

QUERY:
How can blockchain make land records more secure?


RESULT 1

Keyword score: 4
Source: doc2.txt
Chunk ID: chunk_3

Retrieved text:
Punjab's land record system, historically maintained through handwritten patwari registers, has undergone a significant digital transformation through the Land Records Management Information System. Ownership records for millions of parcels have been digitized and made searchable through an online portal, reducing opportunities for record tampering and easing the process of property transfer. A pilot blockchain layer has been introduced to create an immutable audit trail for mutation records, meaning any change to an ownership entry is permanently logged and cryptographically verifiable. This addresses long-standing complaints about fraudulent alterations to land ownership documents. Citizens can now request a copy of their land record, called a fard, from computerized servic

## COMPARISON

QUERY 1:
If we look at query 1, keyword search returned two results. chunk3 got a higher keyword score of 4, while chunk2 got a score of 2. This means chunk3 had more words that matched the query.However, semantic search gave different ranking. chunk2 had a distance of 0.7326, while chunk3 had a distance of 0.838. Since a lower distance means a closer meaning, semantic search considered chunk2 more relevant.This shows that keyword search mainly looks at matching words, while semantic search looks at the meaning of the query. Even though chunk2 had fewer matching words, it was closely related to the topic because its title directly mentions land records and blockchain. This shows how semantic search can find relevant information even when the exact words are not repeated many times.

QUERY 2:
This query also has two results , in keyword search chunk4 has a score of 2 and chunk5  also has a score of 2 both show the same results but in semantic search chunk4 distance is 0.9639 and chunk5 has 1.1003 so chunk4 has a smaller distance so it is considered more semantically similar.In this query both methods ranked chunk4 first and then chunk5.This query shows that keyword and semantic search can produce the same ranking when the query wording matches the document content reasonably well.

Query 3: 
In this query,in keyword search, result 1 shows a keyword score of  5 from chunk 1 and in the second result the keyword score is 4 from chunk3  but if we look  at the retrieved text from both results,the text in result 1 is related to  the query and matches the words but in 2nd result the retrieved text is about land recored not relevant to the query , this shows keyword matching  that may not be the best semantic match.In semantic search result 1 has a distance of 1.0787 from chunk0 and in result 2 distance is 1.0991 from chunk5 so if we look at retrieved text both are conceptually related to the query.Although chunk_0 is more directly related because it specifically discusses digital service delivery and citizen facilitation.